# DPO: The Policy Is Its Own Reward Model

The full code of [DPO: The Policy Is Its Own Reward Model](https://residual-stream.lvtan71.workers.dev/posts/dpo-direct-preference-optimization/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/dpo-direct-preference-optimization) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/dpo-direct-preference-optimization.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 2 min end to end on the RTX 3090 machine that saved the outputs below.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 1 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.14.*" "numpy==2.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "dpo-direct-preference-optimization"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/dpo-direct-preference-optimization


## `dpo_toy.py`

DPO on the exact toy world of the RLHF post: 512 responses, a hidden gold reward, 300 Bradley–Terry comparisons. First the policy family of RLHF's reward model, where DPO and RLHF solved to convergence give the same policy; then one free logit per response, trained with DPO (Adam and SGD), DPO with soft labels, and IPO at two values of β.

In [2]:
!{sys.executable} dpo_toy.py

feature family, beta = 0.5: RLHF policy vs DPO policy, largest difference in any response's probability: 1e-08; KL from ref 2.04, gold gain 2.72

one free logit per response; the 300 comparisons mention 296 of 512 responses, 103 of which only ever won
loss             beta opt   steps     KL   gold  unseen  only won  top 2 chosen log ratio


DPO               0.5 Adam   1000   5.29   1.22    0.00      0.99   0.86     -10.3


DPO               0.5 Adam  30000   5.83   0.35   -0.00      1.00   1.00    -138.4


DPO               0.5 SGD    1000   0.62   0.89    0.15      0.54   0.16       0.3


DPO               0.5 SGD   30000   3.54   1.51    0.00      0.95   0.51      -3.9


...

DPO, soft labels  0.5 Adam  30000   3.76   4.75    0.00      0.91   0.50      -4.1


IPO               0.5 Adam   1000   0.41   0.42    0.18      0.43   0.08       0.1


IPO               0.5 Adam  30000   0.41   0.42    0.18      0.43   0.08       0.1


IPO               0.1 Adam   1000   4.77   0.65    0.00      0.98   0.88      -6.3


IPO               0.1 Adam  30000   4.69   0.66    0.00      0.98   0.91      -6.9


The post printed:

```text
feature family, beta = 0.5: RLHF policy vs DPO policy, largest difference in any response's probability: 1e-08; KL from ref 2.04, gold gain 2.72

one free logit per response; the 300 comparisons mention 296 of 512 responses, 103 of which only ever won
loss             beta opt   steps     KL   gold  unseen  only won  top 2 chosen log ratio
DPO               0.5 Adam   1000   5.29   1.22    0.00      0.99   0.86     -10.3
DPO               0.5 Adam  30000   5.83   0.35    0.00      1.00   1.00    -138.4
DPO               0.5 SGD    1000   0.62   0.89    0.15      0.54   0.16       0.3
DPO               0.5 SGD   30000   3.54   1.51    0.00      0.95   0.51      -3.9
DPO, soft labels  0.5 Adam   1000   3.48   4.51    0.00      0.79   0.50      -3.3
DPO, soft labels  0.5 Adam  30000   3.76   4.75    0.00      0.91   0.50      -4.1
IPO               0.5 Adam   1000   0.41   0.42    0.18      0.43   0.08       0.1
IPO               0.5 Adam  30000   0.41   0.42    0.18      0.43   0.08       0.1
IPO               0.1 Adam   1000   4.77   0.65    0.00      0.98   0.88      -6.3
IPO               0.1 Adam  30000   4.69   0.66    0.00      0.98   0.91      -6.9
```